# Task 3 : Constraint Satisfaction Problems with Propositional Logic

## Requirements

 - Given a m x n matrix, each cell consists of a non-negative integer or it is blank.
 - Each cell has 9 “adjacent” neighbors, including itself and 8 cells around.
 - The player color cells by red or green colors so that the number of green cells which are “adjacent” to a cell matches the number inside.
 - There is no constraint for blank cells.
 - Students solve the given problem using propositional logic and the Glucose3
module of PySAT.
     - Assign a propositional symbol to each cell (true à green, false à red),
     - Enumerate cells to generate CNF clauses representing constraints,
     - Discover the general rule to generate clauses and eliminate redundant
clauses,
     - Find a model satisfying all clauses using Glucose3 of PySAT,
 - Implement a function to evaluate the result matrix and illustrate it on the console
screen with colors.
 - Students organize the program regarding to the OOP model, ensure source code
is compact and reasonable.
 - Recommended editor: Google Colab.

## Import library

In [1]:
!pip install python-sat colorama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.5 MB/s eta 0:00:00


In [2]:
from pysat.solvers import Glucose3
from itertools import combinations
from colorama import Fore, Back, Style
from colorama import just_fix_windows_console
just_fix_windows_console()

## Introduce about PySAT - Glucose3

In [3]:
g = Glucose3()
g.add_clause([1])
g.add_clause([2])
g.add_clause([-1, 3])
g.add_clause([-2, -3, -4])
g.add_clause([-4])

print(g.solve())
# True

g.get_model()
# [1, 2, 3, -4]

True


[1, 2, 3, -4]

## Define Cell Class

In [4]:
class Cell:
    """Class representing a cell in the grid"""
    def __init__(self, row, col, value):
        self.row = row
        self.col = col
        self.value = value  # Integer or None (for blank cells)
        self.is_green = None  # True for green, False for red

    def __str__(self):
        if self.value == '.':
            value_str = '.'
        else:
            value_str = str(self.value)

        if self.is_green is None:
            return value_str
        elif self.is_green:
            return Back.GREEN + value_str + Style.RESET_ALL  # Green background
        else:
            return Back.RED + value_str + Style.RESET_ALL  # Red background

## Define Problem Class

In [5]:
class Problem:
    """Class representing the constraint satisfaction problem"""
    def __init__(self, matrix):
        self.rows = len(matrix)
        self.cols = len(matrix[0]) if matrix else 0
        self.cells = []

        # Initialize cells
        for i in range(self.rows):
            row = []
            for j in range(self.cols):
                row.append(Cell(i, j, matrix[i][j]))
            self.cells.append(row)

    def to_var(self, row, col):
        """Convert cell position to variable number"""
        return row * self.cols + col + 1

    def get_neighbors(self, row, col):
        """Get variable numbers of adjacent cells (including self)"""
        neighbors = []
        for x in range(row - 1, row + 2):
            for y in range(col - 1, col + 2):
                if 0 <= x < self.rows and 0 <= y < self.cols:
                    neighbors.append(self.to_var(x, y))
        return neighbors

    def generate_clauses(self):
        """Generate CNF clauses to represent the constraints"""
        clauses = []

        for i in range(self.rows):
            for j in range(self.cols):
                cell_value = self.cells[i][j].value

                # If cell value is 0, all neighbors (including itself) must be red
                if isinstance(cell_value, int) and cell_value == 0:
                    neighbors = self.get_neighbors(i, j)
                    for neighbor in neighbors:
                        clauses.append([-neighbor])

                # If cell value is > 0, exactly that many adjacent cells must be green
                elif isinstance(cell_value, int) and cell_value > 0:
                    neighbors = self.get_neighbors(i, j)

                    # Using original logic from first implementation
                    combo = list(combinations(neighbors, cell_value))
                    for a in combo:
                        neighbor_copy = neighbors.copy()
                        for x in a:
                            neighbor_copy.remove(x)
                        combo2 = neighbor_copy
                        for x in a:
                            clauses.append([int(h) for h in combo2 + [x]])
                        for y in combo2:
                            temp = [-v for v in a] + [-y]
                            clauses.append([int(g) for g in temp])

        return clauses

## Define Solution Class

In [6]:
class Solution:
    """Class for solving and displaying the solution"""
    def __init__(self, problem):
        self.problem = problem

    def solve(self):
        """Solve the problem using Glucose3 SAT solver"""
        clauses = self.problem.generate_clauses()
        solver = Glucose3()

        # Add clauses to the solver
        for clause in clauses:
            solver.add_clause(clause)

        # Solve the problem
        if solver.solve():
            model = solver.get_model()

            # Update cells with solution
            for i in range(self.problem.rows):
                for j in range(self.problem.cols):
                    var = self.problem.to_var(i, j)
                    if var <= len(model):
                        # Variable is positive in model means green cell
                        self.problem.cells[i][j].is_green = model[var-1] > 0
                    else:
                        # Default to red if somehow not in model
                        self.problem.cells[i][j].is_green = False

            return True
        else:
            return False

    def display(self):
        """Display the solution with colored cells"""
        for i in range(self.problem.rows):
            for j in range(self.problem.cols):
                cell = self.problem.cells[i][j]
                value = str(cell.value) if cell.value != '.' else '.'

                # Force color output
                if cell.is_green:
                    print(Back.GREEN + value, end=' ')
                else:
                    print(Back.RED + value, end=' ')
            print()
        print(Style.RESET_ALL)

In [7]:
def load_matrix_from_file(filename):
    """Load the matrix from a file"""
    with open(filename, 'r') as f:
        lines = f.readlines()
        matrix = []
        for line in lines:
            row = []
            for c in line.strip().split():
                if c == '.':
                    row.append('.')
                else:
                    row.append(int(c))
            matrix.append(row)
        return matrix

In [8]:
def main():
    # Load the matrix from file
    matrix = load_matrix_from_file('input.txt')

    # Create the problem
    problem = Problem(matrix)

    # Create and run the solution
    solution = Solution(problem)
    print("Solving...")
    if solution.solve():
        print("Solution found:")
        print("Green = True cells, Red = False cells")
        solution.display()
    else:
        print("No solution found.")

    # Print confirmation of colorama usage
    print("\nIf colors are not showing, make sure colorama is properly installed.")
    print("Try running: !pip install colorama")
    print("And restart the runtime if needed.")


if __name__ == "__main__":
    main()

Solving...
Solution found:
Green = True cells, Red = False cells
. 2 3 . . 0 . . . . 
. . . . 3 . 2 . . 6 
. . 5 . 5 3 . 5 7 4 
. 4 . 5 . 5 . 6 . 3 
. . 4 . 5 . 6 . . 3 
. . . 2 . 5 . . . . 
4 . 1 . . . 1 1 . . 
4 . 1 . . . 1 . 4 . 
. . . . 6 . . . . 4 
. 4 4 . . . . 4 . . 


If colors are not showing, make sure colorama is properly installed.
Try running: !pip install colorama
And restart the runtime if needed.
